In [ ]:
#claude got 82.30654 (Online score)
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import pygeohash as pgh
import warnings
import os
warnings.filterwarnings('ignore')

# ==========================================
# 1. LOAD DATA
# ==========================================
print("Loading data...")
train = pd.read_csv('../data/dataset/train.csv')
test  = pd.read_csv('../data/dataset/test.csv')

# ==========================================
# 2. CORE TIME FEATURES
# ==========================================
print("Parsing time features...")
def ts_to_min(ts):
    h, m = ts.split(':')
    return int(h) * 60 + int(m)

def add_time_features(df):
    df[['hour', 'minute']] = df['timestamp'].str.split(':', expand=True).astype(int)
    df['slot_min']    = df['timestamp'].apply(ts_to_min)
    df['slot']        = df['slot_min'] // 15          # 0-95: 15-min slot of day
    df['total_minutes'] = df['slot_min']

    # Cyclic encodings
    df['hour_sin']    = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']    = np.cos(2 * np.pi * df['hour'] / 24)
    df['slot_sin']    = np.sin(2 * np.pi * df['slot'] / 96)
    df['slot_cos']    = np.cos(2 * np.pi * df['slot'] / 96)
    df['min_sin']     = np.sin(2 * np.pi * df['minute'] / 60)
    df['min_cos']     = np.cos(2 * np.pi * df['minute'] / 60)

    # Period flags
    df['is_rush']     = df['hour'].isin([7,8,9,17,18,19]).astype(int)
    df['is_night']    = df['hour'].isin([0,1,2,3,4,5]).astype(int)
    df['is_midday']   = df['hour'].isin([11,12,13,14]).astype(int)
    df['is_morning']  = df['hour'].isin([6,7,8,9,10]).astype(int)
    df['day_quarter'] = (df['hour'] // 6).astype(int)
    return df

train = add_time_features(train)
test  = add_time_features(test)

# ==========================================
# 3. GEOHASH DECODING
# ==========================================
print("Decoding geohashes...")
def decode_geohash(df):
    df['geo_p2'] = df['geohash'].str[:2]
    df['geo_p3'] = df['geohash'].str[:3]
    df['geo_p4'] = df['geohash'].str[:4]
    df['geo_p5'] = df['geohash'].str[:5]

    lats, lons = [], []
    for gh in df['geohash']:
        try:
            lat, lon = pgh.decode(gh)
            lats.append(lat); lons.append(lon)
        except:
            lats.append(np.nan); lons.append(np.nan)
    df['latitude']  = lats
    df['longitude'] = lons
    df['lat_lon']   = df['latitude'] * df['longitude']
    return df

train = decode_geohash(train)
test  = decode_geohash(test)

# ==========================================
# 4. TEMPERATURE IMPUTATION
# ==========================================
for df in [train, test]:
    df['Temperature'] = df.groupby('geohash')['Temperature'].transform(
        lambda x: x.fillna(x.median()))
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
    df['Weather']  = df['Weather'].fillna('Unknown')
    df['RoadType'] = df['RoadType'].fillna('Unknown')

# ==========================================
# 5. *** LAG FEATURES *** (The 0.91+ lever)
# ==========================================
print("Building lag features...")

# Split train into day48 and day49
train48 = train[train['day'] == 48].copy()
train49 = train[train['day'] == 49].copy()

# ---- A. Direct lag: same geohash + same timestamp, previous day (day48) ----
lag_d1 = train48.set_index(['geohash', 'timestamp'])['demand']

for df in [train, test]:
    df['lag_d1_same'] = df.set_index(['geohash','timestamp']).index.map(lag_d1)

# For train day48, lag_d1_same is NaN (no day47). Fill with geohash median.
geo_median_demand = train48.groupby('geohash')['demand'].median()
train.loc[train['day']==48, 'lag_d1_same'] = train.loc[train['day']==48].apply(
    lambda r: geo_median_demand.get(r['geohash'], train['demand'].median()), axis=1
)

# ---- B. Adjacent slot lags from day48 ----
# Build a full slot->minute map
all_ts   = pd.concat([train['timestamp'], test['timestamp']]).unique()
min2ts   = {ts_to_min(t): t for t in all_ts}

def add_slot_lag(df, src_df, delta_slots, col_name):
    """Look up demand at (geohash, timestamp ± delta_slots*15min) from src_df."""
    src_idx = src_df.set_index(['geohash', 'timestamp'])['demand']
    shifted_min = (df['slot_min'] + delta_slots * 15).clip(0, 1425)
    shifted_ts  = shifted_min.map(min2ts)
    pairs = pd.MultiIndex.from_arrays([df['geohash'], shifted_ts.fillna('')])
    df[col_name] = src_idx.reindex(pairs).values
    return df

for delta in [-1, +1, -2, +2, -4, +4, -8, +8]:
    col = f'lag_d1_slot{delta:+d}'
    for df in [train, test]:
        add_slot_lag(df, train48, delta, col)

# ---- C. Per-geohash day48 aggregate stats (always available) ----
geo48_agg = train48.groupby('geohash')['demand'].agg(
    geo48_mean='mean', geo48_std='std', geo48_max='max',
    geo48_min='min', geo48_median='median', geo48_q75=lambda x: x.quantile(0.75)
).reset_index()
for df in [train, test]:
    df = df.merge(geo48_agg, on='geohash', how='left')
    # Patch back (merge doesn't modify in place for the outer list)
    # We'll do it properly below
train = train.merge(geo48_agg, on='geohash', how='left')
test  = test.merge(geo48_agg,  on='geohash', how='left')

# Fill missing geo48 stats for new geohashes
for col in ['geo48_mean','geo48_std','geo48_max','geo48_min','geo48_median','geo48_q75']:
    global_fallback = train48['demand'].median()
    train[col] = train[col].fillna(global_fallback)
    test[col]  = test[col].fillna(global_fallback)

# ---- D. Per-geohash per-slot day48 demand (most specific lag) ----
geo_slot48 = train48.groupby(['geohash','slot'])['demand'].mean()
for df in [train, test]:
    df['geo_slot48_mean'] = df.set_index(['geohash','slot']).index.map(geo_slot48)
    df['geo_slot48_mean'] = df['geo_slot48_mean'].fillna(df['geo48_mean'])

# ---- E. Per-geohash per-hour day48 demand ----
geo_hour48 = train48.groupby(['geohash','hour'])['demand'].mean()
for df in [train, test]:
    df['geo_hour48_mean'] = df.set_index(['geohash','hour']).index.map(geo_hour48)
    df['geo_hour48_mean'] = df['geo_hour48_mean'].fillna(df['geo48_mean'])

# ---- F. Day49 early morning context (0:00-2:00) for test rows ----
# train49 has day49 rows at 0:00-2:00 — gives same-day early-morning signal
geo49_early = train49.groupby('geohash')['demand'].agg(
    geo49early_mean='mean', geo49early_max='max', geo49early_last=lambda x: x.iloc[-1]
).reset_index()
test  = test.merge(geo49_early, on='geohash', how='left')
train = train.merge(geo49_early, on='geohash', how='left')
for col in ['geo49early_mean','geo49early_max','geo49early_last']:
    test[col]  = test[col].fillna(test['geo48_mean'])
    train[col] = train[col].fillna(train['geo48_mean'])

# ---- G. Ratio: early-morning day49 vs same-geohash day48 0:00-2:00 ----
# Tells us if today is trending higher/lower than yesterday
geo48_early = train48[train48['slot_min'] <= 120].groupby('geohash')['demand'].mean()
geo48_early.name = 'geo48early_mean'
test['day_ratio']  = test.set_index('geohash').index.map(
    test.groupby('geohash')['geo49early_mean'].first() / (geo48_early + 1e-9)
)
train['day_ratio'] = train.set_index('geohash').index.map(
    train.groupby('geohash')['geo49early_mean'].first() / (geo48_early + 1e-9)
)
for df in [train, test]:
    df['day_ratio'] = df['day_ratio'].fillna(1.0).clip(0.1, 10.0)

print(f"Lag d1 same-slot coverage (test): {test['lag_d1_same'].notna().mean():.2%}")
print(f"Geo48 slot mean coverage (test):  {test['geo_slot48_mean'].notna().mean():.2%}")
print(f"Day49 early morning coverage:     {test['geo49early_mean'].notna().mean():.2%}")

# ==========================================
# 6. LOG TRANSFORM TARGET
# ==========================================
train['demand'] = np.log1p(train['demand'])
global_mean = train['demand'].mean()

# Also log-transform all lag features (they're in demand space)
lag_cols = ['lag_d1_same'] + [f'lag_d1_slot{d:+d}' for d in [-1,+1,-2,+2,-4,+4,-8,+8]] + \
           ['geo48_mean','geo48_median','geo48_max','geo48_min','geo48_q75',
            'geo_slot48_mean','geo_hour48_mean',
            'geo49early_mean','geo49early_max','geo49early_last']

for col in lag_cols:
    for df in [train, test]:
        df[col] = np.log1p(df[col].fillna(0).clip(0))

# ==========================================
# 7. TARGET ENCODING (leak-free, using GLOBAL train)
# ==========================================
print("Target encoding...")

def add_keys(df):
    df['geo_hour']        = df['geohash'] + '_' + df['hour'].astype(str)
    df['geo_slot']        = df['geohash'] + '_' + df['slot'].astype(str)
    df['road_hour']       = df['RoadType'] + '_' + df['hour'].astype(str)
    df['weather_road']    = df['Weather']  + '_' + df['RoadType'].astype(str)
    df['geo_weather']     = df['geohash']  + '_' + df['Weather'].astype(str)
    df['geo_p3_hour']     = df['geo_p3']   + '_' + df['hour'].astype(str)
    df['geo_p4_hour']     = df['geo_p4']   + '_' + df['hour'].astype(str)
    df['geo_rush']        = df['geohash']  + '_' + df['is_rush'].astype(str)
    df['road_weather_hr'] = df['RoadType'] + '_' + df['Weather'] + '_' + df['hour'].astype(str)
    return df

train = add_keys(train)
test  = add_keys(test)

TE_COLS = [
    'geohash','geo_p3','geo_p4',
    'geo_hour','geo_slot','road_hour',
    'weather_road','geo_weather',
    'geo_p3_hour','geo_p4_hour',
    'geo_rush','road_weather_hr','RoadType','Weather',
]

def te_smooth(train_df, test_df, col, target='demand', s=10):
    stats = train_df.groupby(col)[target].agg(['mean','count'])
    stats['smoothed'] = (stats['count']*stats['mean'] + s*global_mean) / (stats['count'] + s)
    train_df[f'{col}_te'] = train_df[col].map(stats['smoothed']).fillna(global_mean)
    test_df[f'{col}_te']  = test_df[col].map(stats['smoothed']).fillna(global_mean)
    return train_df, test_df

for col in TE_COLS:
    train, test = te_smooth(train, test, col)

# Variance signal for top keys
for col in ['geohash','geo_hour','geo_slot']:
    std_map    = train.groupby(col)['demand'].std().fillna(0)
    median_map = train.groupby(col)['demand'].median().fillna(global_mean)
    for df in [train, test]:
        df[f'{col}_std_te']    = df[col].map(std_map).fillna(0)
        df[f'{col}_median_te'] = df[col].map(median_map).fillna(global_mean)

KEY_COLS = ['geo_hour','geo_slot','road_hour','weather_road','geo_weather',
            'geo_p3_hour','geo_p4_hour','geo_rush','road_weather_hr']
for df in [train, test]:
    df.drop(KEY_COLS, axis=1, inplace=True)

# ==========================================
# 8. LABEL ENCODING
# ==========================================
print("Label encoding...")
cat_cols = ['geohash','geo_p2','geo_p3','geo_p4','geo_p5',
            'RoadType','LargeVehicles','Landmarks','Weather']
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]]).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

# ==========================================
# 9. INTERACTION FEATURES
# ==========================================
for df in [train, test]:
    df['lanes_x_hour']     = df['NumberofLanes'] * df['hour']
    df['lanes_x_slot']     = df['NumberofLanes'] * df['slot']
    df['lanes_x_weather']  = df['NumberofLanes'] * df['Weather']
    df['lanes_x_roadtype'] = df['NumberofLanes'] * df['RoadType']
    df['temp_x_hour']      = df['Temperature']   * df['hour']
    df['temp_x_lanes']     = df['Temperature']   * df['NumberofLanes']
    df['temp_bin']         = pd.cut(df['Temperature'], bins=10, labels=False).fillna(0).astype(int)
    df['geo_lat_lon']      = df['latitude']       * df['longitude']
    # Lag ratios (signal: how much does this slot differ from daily avg?)
    df['lag_vs_geo48mean'] = df['lag_d1_same'] - df['geo48_mean']
    df['slot_vs_geo48mean']= df['geo_slot48_mean'] - df['geo48_mean']

# ==========================================
# 10. FEATURE SELECTION & SORT
# ==========================================
# CRITICAL: sort by time for TimeSeriesSplit to work correctly
train = train.sort_values(['day','slot_min']).reset_index(drop=True)

drop_cols = {'Index','timestamp','demand'}
features  = [c for c in train.columns if c not in drop_cols]

X_train = train[features]
y_train = train['demand']
X_test  = test[features]

print(f"\nTotal features: {len(features)}")
print("Lag features:", [f for f in features if 'lag' in f or 'geo48' in f or 'geo49' in f or 'day_ratio' in f])

# ==========================================
# 11. TIMESERIES CV TRAINING
# ==========================================
print("\nTraining with TimeSeriesSplit (n_splits=5)...")
tscv = TimeSeriesSplit(n_splits=5)

lgb_preds = np.zeros(len(X_test))
xgb_preds = np.zeros(len(X_test))
cat_preds = np.zeros(len(X_test))
lgb_scores, xgb_scores, cat_scores = [], [], []

# Weight later folds more — they're closer to test distribution
FOLD_WEIGHTS = np.array([0.5, 0.75, 1.0, 1.5, 2.0])
weight_sum   = FOLD_WEIGHTS.sum()

for fold, (trn_idx, val_idx) in enumerate(tscv.split(X_train)):
    X_tr, X_val = X_train.iloc[trn_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[trn_idx], y_train.iloc[val_idx]
    w = FOLD_WEIGHTS[fold]

    # ---- LightGBM ----
    lgb_model = lgb.LGBMRegressor(
        n_estimators=3000, learning_rate=0.02,
        max_depth=8, num_leaves=127,
        subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=1.0,
        min_child_samples=20,
        random_state=42, verbose=-1,
    )
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(150, verbose=False),
                              lgb.log_evaluation(period=-1)])
    lgb_preds += w * lgb_model.predict(X_test) / weight_sum
    lgb_score  = r2_score(np.expm1(y_val),
                          np.clip(np.expm1(lgb_model.predict(X_val)), 0, None))
    lgb_scores.append(lgb_score)

    # ---- XGBoost ----
    xgb_model = xgb.XGBRegressor(
        n_estimators=3000, learning_rate=0.02,
        max_depth=7, subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=1.0, min_child_weight=5,
        random_state=42, tree_method='hist',
        early_stopping_rounds=150, verbosity=0,
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xgb_preds += w * xgb_model.predict(X_test) / weight_sum
    xgb_score  = r2_score(np.expm1(y_val),
                          np.clip(np.expm1(xgb_model.predict(X_val)), 0, None))
    xgb_scores.append(xgb_score)

    # ---- CatBoost ----
    cat_model = CatBoostRegressor(
        iterations=3000, learning_rate=0.02, depth=8,
        l2_leaf_reg=3, random_strength=1, bagging_temperature=0.5,
        random_seed=42, verbose=False,
    )
    cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val),
                  early_stopping_rounds=150, verbose=False)
    cat_preds += w * cat_model.predict(X_test) / weight_sum
    cat_score  = r2_score(np.expm1(y_val),
                          np.clip(np.expm1(cat_model.predict(X_val)), 0, None))
    cat_scores.append(cat_score)

    print(f"Fold {fold+1} | LGB: {lgb_score:.4f} | XGB: {xgb_score:.4f} | CAT: {cat_score:.4f}")

print("\n========================================")
print(f"AVG LGB R2: {np.mean(lgb_scores):.4f}  |  Fold5: {lgb_scores[-1]:.4f}")
print(f"AVG XGB R2: {np.mean(xgb_scores):.4f}  |  Fold5: {xgb_scores[-1]:.4f}")
print(f"AVG CAT R2: {np.mean(cat_scores):.4f}  |  Fold5: {cat_scores[-1]:.4f}")
print("========================================")

# ==========================================
# 12. BLEND WEIGHTED BY FOLD-5 SCORES
# ==========================================
f5 = np.array([lgb_scores[-1], xgb_scores[-1], cat_scores[-1]])
f5 = np.clip(f5, 0, None)
blend_w = f5 / f5.sum()
print(f"\nBlend weights: LGB={blend_w[0]:.3f}  XGB={blend_w[1]:.3f}  CAT={blend_w[2]:.3f}")

final_log   = blend_w[0]*lgb_preds + blend_w[1]*xgb_preds + blend_w[2]*cat_preds
final_preds = np.clip(np.expm1(final_log), 0, None)

# ==========================================
# 13. SUBMISSION
# ==========================================
submission = pd.DataFrame({'Index': test['Index'], 'demand': final_preds})
submission = submission.sort_values('Index').reset_index(drop=True)

os.makedirs('../submissions', exist_ok=True)
submission.to_csv('../submissions/submission_v3.csv', index=False)
print("\nSubmission saved! Shape:", submission.shape)
print(f"Demand — min: {final_preds.min():.4f} | max: {final_preds.max():.4f} | mean: {final_preds.mean():.4f}")

Loading data...
Parsing time features...
Decoding geohashes...
Building lag features...
Lag d1 same-slot coverage (test): 88.89%
Geo48 slot mean coverage (test):  100.00%
Day49 early morning coverage:     100.00%
Target encoding...
Label encoding...

Total features: 82
Lag features: ['lag_d1_same', 'lag_d1_slot-1', 'lag_d1_slot+1', 'lag_d1_slot-2', 'lag_d1_slot+2', 'lag_d1_slot-4', 'lag_d1_slot+4', 'lag_d1_slot-8', 'lag_d1_slot+8', 'geo48_mean', 'geo48_std', 'geo48_max', 'geo48_min', 'geo48_median', 'geo48_q75', 'geo49early_mean', 'geo49early_max', 'geo49early_last', 'day_ratio', 'lag_vs_geo48mean', 'slot_vs_geo48mean']

Training with TimeSeriesSplit (n_splits=5)...
Fold 1 | LGB: 0.9978 | XGB: 0.9982 | CAT: 0.9952
Fold 2 | LGB: 0.9993 | XGB: 0.9990 | CAT: 0.9989
Fold 3 | LGB: 0.9995 | XGB: 0.9994 | CAT: 0.9990
Fold 4 | LGB: 0.9997 | XGB: 0.9994 | CAT: 0.9992
Fold 5 | LGB: 0.7246 | XGB: 0.7353 | CAT: 0.7112

AVG LGB R2: 0.9442  |  Fold5: 0.7246
AVG XGB R2: 0.9463  |  Fold5: 0.7353
AVG C